### Set-up

In [ ]:
## setting paths 
from pathlib import Path  ### switching to pathlib path-handling instead of the os - should work consistently between HPC and local machine
import sys

# current working directory of the notebook / script
cwd = Path.cwd()

# assume "scripts" folder is one level up from cwd
project_root = cwd.parent.resolve()

if str(project_root) not in sys.path:
    print("Adding to sys.path:", project_root)
    sys.path.append(str(project_root)) # add root to Python path (as a string) for finding scripts modules further



In [ ]:
# loading scripts
from scripts.pre_processing import preprocess_3d_image ### pre-processing function (normalisation + optional downsampling)
from scripts.segment_3d import segment_with_stardist, segment_with_cellpose, segment_cytoplasm ### StarDist3D (3d_demo) segmentation model - light and relatively quick to run
from scripts.io_utils import load_multichannel_images, save_segmentation_results
from scripts.quantification import quantify_objects, sphere_volume_um3, tidy_mask, filter_mask_by_size


In [ ]:
###### INPUT DATA ######

#input_folder = project_root.parent.resolve() / "input_data" / "Images_BBBC050" / "test" / "Images" # put the directory to your input data, relative to the project folder
                                                                                                   # project_root.parent() derives directory above the project folder root
#input_folder = project_root.parent.resolve() / "input_data" / "BBBC035" / "BBBC035_v1_dataset" / "01" 

#input_folder = project_root.parent.resolve() / "input_data" / "2017_07_21_Tom20" / "AICS-11-part13" 

input_folder = project_root.parent.resolve() / "input_data" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF" ### voxel size = (1, 0.48147, 0.48147) (Z, Y, X)


In [ ]:
# 1. load images as multichannel dictionary

channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

voxel_size_um=(1, 0.48147, 0.48147)

all_volumes = load_multichannel_images(input_folder, channel_map)

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

Position001_1
dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
(16, 512, 512)


### MAIN: per-channel preprocessing, segmentation, and quantification

In [48]:
### configs ###
#OUTPUT_DIR = project_root.parent / "output_data" / "Images_BBBC050" / "test" / "Images" 
#OUTPUT_DIR = project_root.parent / "output_data" / "2017_07_21_Tom20" / "AICS-11-part13" 
OUTPUT_DIR = project_root.parent / "output_data" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### params ###
num_test_volumes = 10  # number of volumes from `frames` to segment
use_model = "stardist"  # "cellpose" or "stardist"
diameter = None        # cellpose param: None or specify an estimate, e.g., 20

prob_thresh = 0.4   # stardist param: sets the minimum confidence for accepting a predicted nucleus
nms_thresh = 0.1    # stardist param: remove detections overlapping by more than this threshold


### pre-processing params ###
downsize_factor = 0.5   # scaling factor or 1 to keep original                            
per_slice_norm = False        # True = normalize per-slice, False = normalize whole stack

#### a) nucleus

In [ ]:
# 2. nucleus segmentation

results_dict = {}

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- nuclei preprocessing ---
    nuclei_volume = v["channels"]["nucleus"]
    nuclei_norm = preprocess_3d_image(nuclei_volume, 
                                      downsize_factor=downsize_factor,
                                      apply_gaussian_filter=True,
                                      voxel_size_um=voxel_size_um,
                                      sigma_um=(1.5, 1.5, 1.5))
    
    # --- nuclei segmentation ---
    if use_model == "stardist":
        nuclei_mask = segment_with_stardist(nuclei_norm, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
    elif use_model == "cellpose":
        nuclei_mask = segment_with_cellpose(nuclei_norm, diameter=diameter)
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist

    results_dict[file_name]["nuclei_mask"] = nuclei_mask 
    
    # save nuclei mask (.tiff) as well as PNG overlay image for quality check
    save_segmentation_results(
        nuclei_norm, 
        nuclei_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_nuclei",
        save_overlay=True
    )


[WARN] Blur had little effect on noise.
Noise↓ -443.1%, Edge ratio (mean gradident before / mean gradient after) 0.66
Running StarDist...
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_nuclei_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_nuclei_overlay_MIP.png
[WARN] Blur had little effect on noise.
Noise↓ -98951.7%, Edge ratio (mean gradident before / mean gradient after) 0.63
Running StarDist...
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD127

In [53]:
# (optional): visualise the example nucleus segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(nuclei_norm, name='Preprocessed Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(nuclei_mask, name='Segmentation Mask') ### adding a created mask overlay

<Labels layer 'Segmentation Mask' at 0x1ab57937d60>

In [ ]:
# 2.2 (optional) quantify nuclei

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- retieve nuclei masks ---
    nuclei_mask = results_dict[file_name]["nuclei_mask"]
    nuclei_mask = tidy_mask(nuclei_mask)

    # --- quantify nuclei ---
    df, summary = quantify_objects(nuclei_mask, voxel_size=voxel_size_um, features=("count","volume"))
    print(summary)


{'object_count': 15, 'mean_volume': 458.85136656546}
{'object_count': 6, 'mean_volume': 516.13244804385}
{'object_count': 14, 'mean_volume': 420.6750176446714}
{'object_count': 10, 'mean_volume': 422.78120760942}
{'object_count': 18, 'mean_volume': 398.34550366655}
{'object_count': 11, 'mean_volume': 441.7098222240001}
{'object_count': 11, 'mean_volume': 492.47694826110006}
{'object_count': 14, 'mean_volume': 427.0995593610428}
{'object_count': 16, 'mean_volume': 379.2176817622875}
{'object_count': 17, 'mean_volume': 446.0907228519177}


In [77]:
# 2.2 example thresholds:
expected_d_um = 10.0          # e.g., 10 µm nuclei
tol = 0.25                     # ±25% tolerance
min_um3 = sphere_volume_um3(expected_d_um * (1 - tol))
max_um3 = sphere_volume_um3(expected_d_um * (1 + tol))
print("Expected nuclei volume:", min_um3, "to", max_um3, "µm3")

Expected nuclei volume: 220.8932334555323 to 1022.6538585904274 µm3


In [ ]:
# 2.3 (optional) filtering out nuclei with unwanted size ()
expected_d_um = 10.0
tol = 0.25

for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    # step 1: get mask
    nuclei_mask = results_dict[file_name]["nuclei_mask"]

    # step 2: filter mask
    nuclei_mask_filtered, df = filter_mask_by_size(nuclei_mask, expected_d_um, tol, voxel_size=voxel_size_um, return_df=True)

    # step 3: replace mask in results dict for downstream use
    results_dict[file_name]["nuclei_mask"] = nuclei_mask_filtered

    print(f'{file_name}:',"Before filtering:", len(df))
    print("After filtering:", df["keep"].sum())

Position001_1: Before filtering: 17
After filtering: 15
Position002_2: Before filtering: 6
After filtering: 6
Position003_3: Before filtering: 15
After filtering: 12
Position004_4: Before filtering: 10
After filtering: 10
Position005_5: Before filtering: 20
After filtering: 18
Position006_6: Before filtering: 12
After filtering: 11
Position007_7: Before filtering: 12
After filtering: 11
Position008_8: Before filtering: 14
After filtering: 14
Position009_9: Before filtering: 18
After filtering: 14
Position010_10: Before filtering: 20
After filtering: 17


#### b) cytoplasm

In [ ]:
# 3. cytoplasm segmentation

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    # get the corresponding nuclear mask
    if file_name not in results_dict:
        raise KeyError(f"Nuclear mask for {file_name} not found in previous results.")
    nuclei_mask = results_dict[file_name]["nuclei_mask"]

    # --- cytoplasm channel preprocessing ---
    cytoplasm_volume = v["channels"]['cytoplasm']
    cytoplasm_norm = preprocess_3d_image(
            cytoplasm_volume, 
            downsize_factor=downsize_factor,
            apply_gaussian_filter=False,         # works better for 'intensity' mode of cytoplasm segmentation
            voxel_size_um=voxel_size_um,
            sigma_um=(1, 1, 1)        
            )
    
    # --- cytoplasm segmentation (watershed) ---
    cytoplasm_mask = segment_cytoplasm(
        nuclei_mask,
        cytoplasm_norm, 
        mode="intensity",  # could also be "membrane" for alternative approach
        membrane_threshold=0.1
    )
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["cytoplasm_mask"] = cytoplasm_mask


    # saving results into a mask (.tiff) as well as PNG overlay image for quality check
    save_segmentation_results(
        cytoplasm_norm, 
        cytoplasm_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_cytoplasm",
        save_overlay=True
    )

[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_cytoplasm_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_cytoplasm_overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position002_2_cytoplasm_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position002_2_cytoplasm_overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position003_3_cytoplasm_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position

In [17]:
# (optional): visualise the example cytoplasm segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(cytoplasm_norm, name='Cyto Preprocessed Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(cytoplasm_mask, name='Cyto Segmentation Mask') ### adding a created mask overlay
viewer.add_image(nuclei_norm, name='Nuc Preprocessed Image', rendering='mip', colormap='gray') ### adding a created mask overlay
viewer.add_labels(nuclei_mask, name='Nuc Segmentation Mask') ### adding a created mask overlay

<Labels layer 'Nuc Segmentation Mask' at 0x26634408730>

#### c) organelle

##### 4. segmenting/quantifying intracellular structures (per cell mask) 
 


In [54]:
# 4.0 obtain the suggested parameters for the preprocessing

from aicssegmentation.core.pre_processing_utils import suggest_normalization_param
from aicssegmentation.core.pre_processing_utils import intensity_normalization, image_smoothing_gaussian_3d

### functions from aicssegmentation 
structure_img = all_volumes[0]["channels"]["intracellular"]
suggest_normalization_param(structure_img)


mean intensity of the stack: 1746.7548496723175
the standard deviation of intensity of the stack: 2738.485222180432
0.9999 percentile of the stack intensity is: 45461.127299994696
minimum intensity of the stack: 575
maximum intensity of the stack: 65535
suggested upper range is 16.0, which is 45562.51840455923
suggested lower range is 0.0, which is 1746.7548496723175
So, suggested parameter for normalization is [0.0, 16.0]
To further enhance the contrast: You may increase the first value (may loss some dim parts), or decrease the second value(may loss some texture in super bright regions)
To slightly reduce the contrast: You may decrease the first value, or increase the second value


About selected algorithms and tuned parameters

- **Intensity normalization**: Parameter intensity_scaling_param has two options: two values, say [A, B], or single value, say [K]. For the first case, A and B are non-negative values indicating that the full intensity range of the stack will first be cut-off into **[mean - A * std, mean + B * std]** and then rescaled to **[0, 1]**. The smaller the values of A and B are, the higher the contrast will be. For the second case, K>0 indicates min-max Normalization with an absolute intensity upper bound K (i.e., anything above K will be chopped off and reset as the minimum intensity of the stack) and K=0 means min-max Normalization without any intensity bound.

    - Parameter for fibrillarin: intensity_scaling_param = [0.5, 18]

    - Parameter for beta catenin: intensity_scaling_param = [4, 27]

- **Smoothing**

    3D gaussian smoothing with gaussian_smoothing_sigma = 1 (same for fibrillarin and beta catenin). The large the value is, the more the image will be smoothed.

[from AllenCell demo notebook: https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb] 

In [ ]:
# 4.1 do recommended pre-processing of the channel

### example: segmentation workflow for spotty structures [based on https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb]
# suggested parameter for normalization is [0.0, 17.0] (previous step)

##### params #####
intensity_scaling_param = [0, 17]
gaussian_smoothing_sigma = 1

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    struct_img0 = v["channels"]["intracellular"]
    
    # --- preprocessing ---
    # intensity normalization
    struct_img_norm = intensity_normalization(struct_img0, scaling_param=intensity_scaling_param)
    # making sure the image has the same shape as cytoplasm (from downsizing as the cytoplasm) -- TODO: needs fixing. remove exessive preprocessing, when no downsizing applied
    struct_img_norm = preprocess_3d_image(          
            struct_img_norm, 
            downsize_factor=downsize_factor,
            apply_gaussian_filter=False,         
            voxel_size_um=voxel_size_um,        
            )    

    # smoothing with gaussian filter
    structure_img_smooth = image_smoothing_gaussian_3d(struct_img_norm, sigma=gaussian_smoothing_sigma)

    # saving results in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["structure_img_smooth"] = structure_img_smooth


In [ ]:
# (optional) visualise the processed structure (last file)
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
#viewer.add_image(struct_img0, name='Intra Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_image(struct_img_norm, name='Intra Normalised Image', rendering='mip', colormap='gray') ### adding a preprocessed image overlay
viewer.add_image(structure_img_smooth, name='Intra Preprocessed Image', rendering='mip', colormap='gray') ### adding a preprocessed image overlay
viewer.add_labels(nuclei_mask, name='Nucleus Segmentation Mask') ### adding a created mask overlay


<Labels layer 'Nuc Segmentation Mask' at 0x265ef402560>

##### apply 2d spot filter
- Parameter syntax: [[scale_1, cutoff_1], [scale_2, cutoff_2], ....]

    - scale_x is set based on the estimated radius of your target spotty shape. For example, if visually the diameter of the spotty objects is usually 3~4 pixels, then you may want to set scale_x as 1 or something near 1 (like 1.25). Multiple scales can be used, if you have objects of very different sizes.
    - cutoff_x is a threshold applied on the actual filter reponse to get the binary result. Smaller cutoff_x may yielf fatter segmentation, while larger cutoff_x could be less permisive and yield less objects and slimmer segmentation.

Examples:
- Parameter for fibrillarin: s2_param = [[1, 0.01]]

- Parameter for beta catenin: s2_param = [[1.5, 0.01]]

In [ ]:
# 4.2 apply recommended algorithms to the channel (e.g. dot_2d_slice_by_slice_wrapper or filament_2d_wrapper for spotty structures) - AllenCell based
from aicssegmentation.core.seg_dot import dot_2d_slice_by_slice_wrapper
from skimage.morphology import remove_small_objects, binary_closing, ball , dilation

##### params #####
s2_param = [[0.75, 0.01]]
minArea = 5 # example of the smallest object size to detect

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    # retrieve the channel
    structure_img_smooth = results_dict[file_name]["structure_img_smooth"]
    
    # apply 2d spot filter (returns binary mask)
    structure_mask = dot_2d_slice_by_slice_wrapper(structure_img_smooth, s2_param) # binary but not labeled mask

    # post-process the image: remove small objects if needed
    #structure_mask = remove_small_objects(structure_mask>0, min_size=minArea, connectivity=1, in_place=False)


    #append and save results
    results_dict[file_name]["structure_mask"] = structure_mask

    # saving results into a mask (.tiff) as well as PNG overlay image
    save_segmentation_results(
        structure_img_smooth, 
        structure_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_structure",
        save_overlay=True
    )



[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_structure_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position001_1_structure_overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position002_2_structure_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position002_2_structure_overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position003_3_structure_mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position

In [ ]:
# (optional) visualise the processed structure (the latest file)
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
#viewer.add_image(struct_img0, name='Intra Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_image(structure_img_smooth, name='Intra Preprocessed Image', rendering='mip', colormap='gray') ### adding a preprocessed image overlay
viewer.add_image(structure_mask, name='BW') ### adding a created bw overlay


<Image layer 'BW' at 0x265f7739390>

In [ ]:
# 4.3 quantify the intracellular channel in respect to the cellular masks 
# TODO: add alternatives of quantifying per cell overall, nuclei only or cytoplasm only

from scripts.quantification import quantify_structures_per_cell

##### params #####
return_level = 'both'   # whether to return results at the "object" level, "cell" level, or "both"


# --- analyse all images ---

all_results_cell = []  # store all per-volume results
all_results_obj = []

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- get the corresponding masks and channels ---
    if file_name not in results_dict:
        raise KeyError(f"Masks for {file_name} not found in previous results.")
    
    #nuclei_mask = results_dict[file_name]["nuclei_mask"]       ### could be used in case calculation needs to be done per area of nucleus
    cytoplasm_mask = results_dict[file_name]["cytoplasm_mask"]
    structure_mask = results_dict[file_name]["structure_mask"]
    structure_img_smooth = results_dict[file_name]["structure_img_smooth"]


    # --- mapping intracellular structure channel to the previously obtained masks and quantifying ---
    df_obj, df_cells = quantify_structures_per_cell(
        cytoplasm_mask=cytoplasm_mask,      # cytoplasm_mask: labeled cells           TODO: add alternatives 
        object_mask=structure_mask,       # struct_mask: binary mask, e.g. from dot_2d_slice_by_slice_wrapper
        intensity_img=structure_img_smooth,     # struct_intensity: original preprocessed image used to detect spots
        voxel_size=voxel_size_um,
        features=("count","volume","intensity"), # which features to calculate
        return_level = return_level              # whether to return results at the "object" level, "cell" level, or "both" (in case of the latter: returns df_obj, df_cells)
    )

    # --- add metadata ---
     # add metadata: filename + space for experimental info if any
    df_cells["filename"] = file_name
    #df_obj["filename"] = file_name

    # --- store together results for all images ---
    all_results_cell.append(df_cells)
    #all_results_obj.append(df_obj)


# concatenate into one dataframe
df_all = pd.concat(all_results_cell, ignore_index=True)
#df_all_obj = pd.concat(all_results_obj, ignore_index=True)


# save to csv
output_path = OUTPUT_DIR / f"{return_level}_quantification_results.csv"
df_all.to_csv(output_path, index=False)

print(f"Results exported to {output_path}")
print(df_all.head())
#print(df_all_obj.head())

Results exported to C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\both_quantification_results.csv
   cell_id  cell_volume  object_count  total_object_volume  \
0        1  5086.448765            87           267.976245   
1        2   223.468080             1            22.254083   
2        3  4054.647496            59           148.360551   
3        4   643.977517            10            66.994061   
4        5  2747.220140            41           135.610816   

   object_volume_fraction  mean_volume_per_cell_volume  mean_object_intensity  \
0                0.052684                     0.000606               0.097810   
1                0.099585                     0.099585               0.138494   
2                0.036590                     0.000620               0.096114   
3                0.104032                     0.010403               0.112847   
4                0.049363                     0.001204      

In [ ]:
# example quality check after you created df_obj (with 'label' and 'cell_id')
from skimage.measure import label
from scripts.quantification import save_struct_mip_overlay_by_cell

# make sure struct_mask is labeled
if np.array_equal(np.unique(structure_mask), [0,1]) or structure_mask.dtype == bool:
    struct_labeled = label(structure_mask)
else:
    struct_labeled = structure_mask.copy()

save_struct_mip_overlay_by_cell(
    struct_labeled=struct_labeled,      # NOT the cytoplasm mask
    df_obj=df_obj,
    raw_volume=structure_img_smooth,       # or original structure channel
    out_png=OUTPUT_DIR / f"{file_name}_struct_to_cell_MIP.png"
)


Unique cell ids in df_obj: [ 0  1  2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
Unique values in mapped mask: [ 0  1  2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[INFO] Saved structure→cell overlay MIP to C:\Users\Lada\Desktop\job\tsn_rocks\output_data\S-BIAD1272_30min_stimulation\240109_240110_S1_30min_pMAPK_EGF\Position010_10_struct_to_cell_MIP.png


#### Set-up the config file for further script-based batch processing